In [ ]:
import pygame
import numpy as np
import random
from pygame.locals import *

# Game configuration constants
WINDOW_SIZE = 800  # Window size
MAZE_SIZE = 10     # Maze grid size (10×10, adjustable to 15/20)
CELL_SIZE = WINDOW_SIZE // (MAZE_SIZE + 2)  # Grid cell size (with margin)
FPS = 60           # Animation frame rate
CONSECUTIVE_SUCCESS = 50  # Consecutive successful episodes to stop training
OPTIMAL_STEP_THRESHOLD = MAZE_SIZE * 2  # Optimal step threshold (adjust based on maze)

# Color definitions
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
GREEN = (0, 255, 0)  # Start point
RED = (255, 0, 0)    # End point
BLUE = (0, 0, 255)   # Agent
LIGHT_BLUE = (173, 216, 230)  # Movement path
GRAY = (128, 128, 128)        # Info panel background
YELLOW = (255, 255, 0)        # Pause/end prompt

# Initialize Pygame
pygame.init()
screen = pygame.display.set_mode((WINDOW_SIZE + 200, WINDOW_SIZE))  # Extra space for info panel on right
pygame.display.set_caption("Q-Learning Maze Solver (Stoppable)")
clock = pygame.time.Clock()
font = pygame.font.SysFont("Arial", 20)  # English font for better compatibility
small_font = pygame.font.SysFont("Arial", 16)

class Maze:
    """Maze Class: Generates random maze using Prim's algorithm, ensuring unique path from start to end"""
    def __init__(self, size):
        self.size = size
        self.maze = self._generate_prim_maze()
        self.start = (0, 0)  # Start point: top-left corner
        self.end = (size-1, size-1)  # End point: bottom-right corner
        self.maze[self.start[0]][self.start[1]] = 0  # Start point is passable
        self.maze[self.end[0]][self.end[1]] = 0      # End point is passable

    def _generate_prim_maze(self):
        """Generate maze using Prim's algorithm: 0=passable, 1=wall"""
        maze = np.ones((self.size, self.size), dtype=int)
        start_x, start_y = random.randint(0, self.size-1), random.randint(0, self.size-1)
        maze[start_x][start_y] = 0
        walls = []
        # Initialize walls around start point
        for dx, dy in [(-1,0), (1,0), (0,-1), (0,1)]:
            x, y = start_x+dx, start_y+dy
            if 0<=x<self.size and 0<=y<self.size:
                walls.append((x, y, start_x, start_y))
        # Iteratively generate maze
        while walls:
            x, y, px, py = random.choice(walls)
            walls.remove((x, y, px, py))
            if maze[x][y] == 1:
                maze[x][y] = 0
                maze[(x+px)//2][(y+py)//2] = 0  # Break through wall
                # Add new walls
                for dx, dy in [(-1,0), (1,0), (0,-1), (0,1)]:
                    nx, ny = x+dx, y+dy
                    if 0<=nx<self.size and 0<=ny<self.size and maze[nx][ny] == 1:
                        walls.append((nx, ny, x, y))
        return maze

    def is_valid(self, x, y):
        """Check if coordinate (x,y) is valid (within bounds + not a wall)"""
        return 0<=x<self.size and 0<=y<self.size and self.maze[x][y] == 0

    def draw(self, screen):
        """Draw maze: walls, start point, end point"""
        screen.fill(WHITE)
        # Draw grid
        for x in range(self.size):
            for y in range(self.size):
                rect = Rect(
                    CELL_SIZE + y*CELL_SIZE,
                    CELL_SIZE + x*CELL_SIZE,
                    CELL_SIZE-1, CELL_SIZE-1  # -1 for grid lines
                )
                if self.maze[x][y] == 1:
                    pygame.draw.rect(screen, BLACK, rect)  # Wall
                elif (x, y) == self.start:
                    pygame.draw.rect(screen, GREEN, rect)  # Start point
                elif (x, y) == self.end:
                    pygame.draw.rect(screen, RED, rect)    # End point

class QLearningAgent:
    """Q-Learning Agent Class: Manages Q-table, action selection, and Q-value updates"""
    def __init__(self, maze_size, lr=0.1, gamma=0.9, epsilon=0.9):
        self.maze_size = maze_size
        self.num_states = maze_size * maze_size  # Total states: number of grid cells
        self.num_actions = 4  # Action space: up(0), down(1), left(2), right(3)
        self.lr = lr          # Learning rate
        self.gamma = gamma    # Discount factor
        self.epsilon = epsilon  # Exploration rate
        self.epsilon_min = 0.1  # Minimum exploration rate
        self.epsilon_decay = 0.0008  # Exploration rate decay rate
        # Initialize Q-table: [number of states, number of actions], all zeros
        self.q_table = np.zeros((self.num_states, self.num_actions))

    def state_encoder(self, x, y):
        """Encode grid coordinates (x,y) to integer state: x*size + y"""
        return x * self.maze_size + y

    def choose_action(self, x, y, maze):
        """Select action using epsilon-greedy strategy: prioritize exploitation, random exploration"""
        state = self.state_encoder(x, y)
        # Filter valid actions (no wall collision)
        valid_actions = []
        for action in range(self.num_actions):
            nx, ny = self._action2coord(x, y, action)
            if maze.is_valid(nx, ny):
                valid_actions.append(action)
        # Greedy/exploration selection
        if random.uniform(0, 1) < self.epsilon:
            # Exploitation: choose action with max Q-value
            q_vals = self.q_table[state][valid_actions]
            max_idx = np.argmax(q_vals)
            action = valid_actions[max_idx]
        else:
            # Exploration: randomly choose valid action
            action = random.choice(valid_actions) if valid_actions else 0
        return action, valid_actions

    def _action2coord(self, x, y, action):
        """Convert action to coordinates: up(0), down(1), left(2), right(3)"""
        if action == 0: x -= 1
        elif action == 1: x += 1
        elif action == 2: y -= 1
        elif action == 3: y += 1
        return x, y

    def update_q(self, x, y, action, reward, nx, ny, done, maze):
        """Update Q-value using Bellman equation"""
        state = self.state_encoder(x, y)
        next_state = self.state_encoder(nx, ny) if maze.is_valid(nx, ny) else state
        # Calculate target Q-value
        if done:
            target = reward  # No future reward when game ends
        else:
            target = reward + self.gamma * np.max(self.q_table[next_state])
        # Update Q-value
        self.q_table[state][action] += self.lr * (target - self.q_table[state][action])
        # Linear decay of exploration rate
        if self.epsilon > self.epsilon_min:
            self.epsilon -= self.epsilon_decay

    def save_q_table(self):
        """Save trained Q-table to local file"""
        np.save("q_table_maze.npy", self.q_table)
        print("Q-table saved to: q_table_maze.npy")

    def load_q_table(self):
        """Load local Q-table (for testing mode)"""
        try:
            self.q_table = np.load("q_table_maze.npy")
            self.epsilon = 0  # Testing mode: no exploration, pure exploitation
            print("Successfully loaded Q-table, entering test mode!")
        except FileNotFoundError:
            print("Q-table file not found, starting training from scratch!")

class Game:
    """Game Main Class: Manages game flow, interaction, rendering, reward calculation (added stop/pause logic)"""
    def __init__(self):
        self.maze = Maze(MAZE_SIZE)
        self.agent = QLearningAgent(MAZE_SIZE)
        self.agent_x, self.agent_y = self.maze.start  # Agent initial position
        self.visited = set()  # Track visited states to prevent loops
        self.visited.add((self.agent_x, self.agent_y))
        self.episode = 1  # Training episode count
        self.step = 0     # Current episode step count
        self.step_limit = 200  # Maximum steps per episode
        self.cumulative_reward = 0  # Cumulative reward for current episode
        self.training_mode = True   # Initial mode: training
        self.game_done = False      # Whether current episode is done
        self.path = [(self.agent_x, self.agent_y)]  # Agent movement path
        self.paused = False         # Pause state
        self.training_completed = False  # Training completion state
        self.consecutive_success_count = 0  # Consecutive successful episodes count

    def calculate_reward(self, nx, ny):
        """Reward function: core mechanism to guide agent behavior"""
        # Reach end point: +100 (maximum positive reward)
        if (nx, ny) == self.maze.end:
            return 100
        # Wall collision (invalid action): -10 (negative penalty)
        if not self.maze.is_valid(nx, ny):
            return -10
        # Repeat state (looping): -5 (negative penalty)
        if (nx, ny) in self.visited:
            return -5
        # Valid move per step: -1 (small penalty to encourage shortest path)
        return -1

    def reset_episode(self):
        """Reset current episode: agent returns to start, clear states"""
        self.agent_x, self.agent_y = self.maze.start
        self.visited = set()
        self.visited.add((self.agent_x, self.agent_y))
        self.step = 0
        self.cumulative_reward = 0
        self.game_done = False
        self.path = [(self.agent_x, self.agent_y)]

    def check_training_complete(self):
        """Check if training is complete: N consecutive successful episodes with optimal steps"""
        if (self.agent_x, self.agent_y) == self.maze.end and self.step <= OPTIMAL_STEP_THRESHOLD:
            self.consecutive_success_count += 1
            # Training complete when consecutive success threshold is reached
            if self.consecutive_success_count >= CONSECUTIVE_SUCCESS:
                self.training_completed = True
                self.agent.save_q_table()
                print(f"Training completed! Total episodes: {self.episode}, {CONSECUTIVE_SUCCESS} consecutive optimal solves")
        else:
            self.consecutive_success_count = 0  # Reset consecutive success count

    def step_game(self):
        """Execute single game step: agent selects action → move → calculate reward → update Q-table (add training termination check)"""
        if self.game_done:
            # Check if training is complete
            self.check_training_complete()
            self.reset_episode()
            self.episode += 1
            return
        if self.paused or self.training_completed:
            return  # Do not execute game step when paused/training completed

        # Agent selects action
        action, _ = self.agent.choose_action(self.agent_x, self.agent_y, self.maze)
        # Execute action and calculate new coordinates
        nx, ny = self.agent._action2coord(self.agent_x, self.agent_y, action)
        # Calculate immediate reward
        reward = self.calculate_reward(nx, ny)
        self.cumulative_reward += reward
        self.step += 1
        # Determine if current episode is done (reach end point/max steps)
        if (nx, ny) == self.maze.end:
            self.game_done = True
            reward = 100
            self.cumulative_reward += reward
        elif self.step >= self.step_limit:
            self.game_done = True
        # Update position and path for valid moves
        if self.maze.is_valid(nx, ny):
            self.agent_x, self.agent_y = nx, ny
            self.visited.add((nx, ny))
            self.path.append((nx, ny))
        # Update Q-table in training mode
        if self.training_mode:
            self.agent.update_q(self.agent_x, self.agent_y, action, reward, nx, ny, self.game_done, self.maze)

    def draw_agent_and_path(self, screen):
        """Draw agent and movement path"""
        # Draw path
        for (x, y) in self.path:
            rect = Rect(
                CELL_SIZE + y*CELL_SIZE + CELL_SIZE//4,
                CELL_SIZE + x*CELL_SIZE + CELL_SIZE//4,
                CELL_SIZE//2, CELL_SIZE//2
            )
            pygame.draw.rect(screen, LIGHT_BLUE, rect, border_radius=5)
        # Draw agent (blue circle)
        agent_x = CELL_SIZE + self.agent_y*CELL_SIZE + CELL_SIZE//2
        agent_y = CELL_SIZE + self.agent_x*CELL_SIZE + CELL_SIZE//2
        pygame.draw.circle(screen, BLUE, (agent_x, agent_y), CELL_SIZE//3)

    def draw_info_panel(self, screen):
        """Draw right-side info panel: training state, steps, reward, etc. (added pause/end prompts)"""
        # Panel background
        panel_rect = Rect(WINDOW_SIZE, 0, 200, WINDOW_SIZE)
        pygame.draw.rect(screen, GRAY, panel_rect)
        # Draw text info
        info = [
            f"Mode: {'Training' if self.training_mode else 'Testing'}",
            f"Episode: {self.episode}",
            f"Steps: {self.step}/{self.step_limit}",
            f"Cumulative Reward: {self.cumulative_reward:.0f}",
            f"Agent Position: ({self.agent_x},{self.agent_y})",
            f"Exploration Rate: {self.agent.epsilon:.2f}",
            f"Consecutive Success: {self.consecutive_success_count}/{CONSECUTIVE_SUCCESS}",
            "=== Shortcuts ===",
            "T: Toggle Train/Test",
            "S: Save Q-table",
            "L: Load Q-table",
            "R: Reset Episode",
            "P: Pause/Resume",
            "ESC: Terminate Training",
            "Q: Quit Game"
        ]
        y = 20
        for text in info:
            surf = small_font.render(text, True, WHITE)
            screen.blit(surf, (WINDOW_SIZE + 10, y))
            y += 30
        # State prompts
        if self.paused:
            surf = font.render("Paused! (Press P to resume)", True, YELLOW)
            screen.blit(surf, (WINDOW_SIZE + 10, WINDOW_SIZE - 90))
        elif self.training_completed:
            surf = font.render("Training Complete!", True, GREEN)
            screen.blit(surf, (WINDOW_SIZE + 10, WINDOW_SIZE - 90))
        elif self.game_done:
            if (self.agent_x, self.agent_y) == self.maze.end:
                surf = font.render("Solved!", True, GREEN)
            else:
                surf = font.render("Step Limit Reached!", True, RED)
            screen.blit(surf, (WINDOW_SIZE + 10, WINDOW_SIZE - 60))

    def handle_events(self):
        """Handle keyboard events: added pause/terminate training logic"""
        for event in pygame.event.get():
            if event.type == QUIT:
                pygame.quit()
                exit()
            if event.type == KEYDOWN:
                if event.key == K_q:  # Quit game
                    pygame.quit()
                    exit()
                elif event.key == K_r:  # Reset current episode
                    self.reset_episode()
                elif event.key == K_t:  # Toggle training/testing mode
                    self.training_mode = not self.training_mode
                    if not self.training_mode:
                        self.agent.epsilon = 0
                    print(f"Switched to: {'Testing Mode' if not self.training_mode else 'Training Mode'}")
                elif event.key == K_s:  # Save Q-table
                    self.agent.save_q_table()
                elif event.key == K_l:  # Load Q-table
                    self.agent.load_q_table()
                elif event.key == K_p:  # Pause/resume
                    self.paused = not self.paused
                    print(f"Game {'paused' if self.paused else 'resumed'}")
                elif event.key == K_ESCAPE:  # Terminate training (manual)
                    self.training_completed = True
                    self.agent.save_q_table()
                    print("Training terminated manually, Q-table saved!")

    def run(self):
        """Game main loop: added pause/training completion check, no longer infinite"""
        while True:
            self.handle_events()
            self.step_game()
            # Render
            self.maze.draw(screen)
            self.draw_agent_and_path(screen)
            self.draw_info_panel(screen)

            # Show final prompt when training completed
            if self.training_completed:
                final_text = f"Training Complete! Optimal Steps: {self.step} Episodes: {self.episode}"
                surf = font.render(final_text, True, GREEN)
                screen.blit(surf, (CELL_SIZE, CELL_SIZE // 2))

            # Update display
            pygame.display.update()
            clock.tick(FPS)

# Start game
if __name__ == "__main__":
    game = Game()
    game.run()

pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


error: display Surface quit

: 